---
title: "Exam task: CITE-seq analysis of a COVID-19 PBMC dataset"
author: 
  - Maximilian Nuber
  - Junyan Lu
format:
  html:
    toc: true
    code-fold: false
    embed-resources: true
execute:
  echo: true
  warning: false
  message: false
jupyter: python3
---

# Exam task: CITE-seq analysis of a COVID-19 PBMC dataset

In this task, you will analyze a geometrically sketched subset of a public COVID-19 CITE-seq PBMC dataset. The dataset contains paired RNA and antibody-derived tag (ADT) measurements for the same cells.

The goal is not to reproduce the full preprocessing pipeline. Instead, you should use the prepared RNA and ADT objects to perform a compact multi-modal analysis, annotate major immune cell populations, and interpret how RNA and protein measurements complement each other.

You are expected to submit an executed notebook containing code, plots, and short written interpretations.

## Learning goals

By the end of this task, you should be able to:

1. Load paired RNA and ADT single-cell objects.
2. Verify that both modalities contain the same cells in the same order.
3. Perform RNA-based dimensionality reduction, clustering, and marker inspection.
4. Use ADT markers to support or refine cell-type annotation.
5. Discuss how RNA and protein measurements agree or disagree.
6. Interpret immune-cell differences in the context of COVID-19 metadata.

In [ ]:
#| eval: false
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Load the data

The RNA and ADT objects have already been subset using geometric sketching. This keeps the dataset small enough for an exam while preserving the broad structure of the full dataset.

In [ ]:
#| eval: false
rna = sc.read_h5ad("../data/exam_rna_geosketch_30k.h5ad")
adt = sc.read_h5ad("../data/exam_adt_geosketch_30k.h5ad")

rna

In [ ]:
#| eval: false
adt

The RNA and ADT objects should contain the same cells in the same order.

In [ ]:
#| eval: false
assert rna.n_obs == adt.n_obs
assert (rna.obs_names == adt.obs_names).all()

## Inspect available metadata

Look at the available metadata columns. Identify columns that describe sample identity, disease status, disease severity, donor, or cell-type annotations if present.

In [ ]:
#| eval: false
rna.obs.columns

In [ ]:
#| eval: false
rna.obs.head()

### Task 1: Metadata overview

Write a short paragraph describing the dataset based on the metadata.

Your answer should mention:

- how many cells are present;
- which metadata columns are relevant for biological interpretation;
- which column you will use as sample or patient identity;
- which column you will use as disease or severity group, if available.

In [ ]:
#| eval: false
# Your code here

**Your interpretation:**  
Write 3–5 sentences here.

## RNA analysis

Use the RNA object to compute a standard single-cell embedding and clusters.

The object may already contain normalized values, but for this task you may assume that the RNA matrix is suitable for exploratory dimensionality reduction and visualization.

In [ ]:
#| eval: false
# Optional: inspect whether X looks log-normalized or count-like
rna.X

In [ ]:
#| eval: false
sc.pp.highly_variable_genes(rna, n_top_genes=3000)
sc.pp.pca(rna, n_comps=50, mask_var="highly_variable")
sc.pp.neighbors(rna, n_pcs=30)
sc.tl.umap(rna)
sc.tl.leiden(rna, resolution=0.8)

In [ ]:
#| eval: false
sc.pl.umap(rna, color=["leiden"], frameon=False)

### Task 2: RNA-based structure

Make UMAP plots colored by:

- cluster assignment;
- sample or donor;
- disease status or severity, if available;
- any existing cell-type annotation, if available.

Then briefly describe whether the main structure appears to reflect cell type, sample identity, disease status, or a combination of these.

# Replace these column names with the relevant columns from rna.obs
columns_to_plot = [
    "leiden",
    # "sample",
    # "disease_status",
    # "severity",
    # "cell_type",
]

sc.pl.umap(rna, color=columns_to_plot, frameon=False, wspace=0.4)

**Your interpretation:**  
Write 4–6 sentences here.

## RNA marker genes

Use canonical RNA markers to identify major immune cell populations.

Examples of useful markers:

- T cells: `CD3D`, `CD3E`, `TRAC`
- CD4 T cells: `CD4`, `IL7R`, `CCR7`
- CD8 T cells: `CD8A`, `CD8B`, `GZMK`, `NKG7`
- NK cells: `NKG7`, `GNLY`, `KLRD1`, `FCGR3A`
- B cells: `MS4A1`, `CD79A`, `CD74`
- Plasma cells: `MZB1`, `JCHAIN`, `XBP1`
- Monocytes: `LYZ`, `S100A8`, `S100A9`, `FCGR3A`, `MS4A7`
- Dendritic cells: `FCER1A`, `CST3`, `CLEC9A`, `LILRA4`
- Platelets: `PPBP`, `PF4`

In [ ]:
#| eval: false
rna_markers = [
    "CD3D", "CD3E", "TRAC",
    "CD4", "IL7R", "CCR7",
    "CD8A", "CD8B", "NKG7", "GNLY",
    "MS4A1", "CD79A",
    "LYZ", "S100A8", "S100A9", "FCGR3A", "MS4A7",
    "FCER1A", "CST3", "LILRA4",
    "PPBP", "PF4",
]

rna_markers = [g for g in rna_markers if g in rna.var_names]
rna_markers

In [ ]:
#| eval: false
sc.pl.umap(
    rna,
    color=rna_markers,
    frameon=False,
    ncols=4,
    vmax="p99",
)

### Task 3: RNA-based cell-type annotation

Use the RNA marker plots to assign broad cell-type labels to the main Leiden clusters.

Create a new column:

```python
rna.obs["manual_cell_type"] = ...

In [ ]:
#| eval: false
# Example structure. Replace cluster IDs and labels with your own interpretation.
cluster_to_cell_type = {
    # "0": "CD4 T cell",
    # "1": "Monocyte",
    # "2": "B cell",
}

rna.obs["manual_cell_type"] = (
    rna.obs["leiden"]
    .astype(str)
    .map(cluster_to_cell_type)
    .fillna("Unassigned")
)

sc.pl.umap(rna, color="manual_cell_type", frameon=False)

**Your interpretation:**  
Explain which markers you used for at least three major cell types.

## ADT analysis

The ADT object contains antibody-derived tag counts for the same cells. ADT measurements are useful because they directly measure surface proteins, which can support or refine RNA-based annotations.

In [ ]:
#| eval: false
adt

In [ ]:
#| eval: false
adt.var_names[:20]

Normalize the ADT data. A common simple choice is centered log-ratio normalization. For this task, use Scanpy's total-count normalization and log transform for visualization, unless you prefer to implement CLR normalization yourself.

In [ ]:
#| eval: false
adt_norm = adt.copy()

sc.pp.normalize_total(adt_norm, target_sum=1e4)
sc.pp.log1p(adt_norm)

Copy the RNA UMAP coordinates into the ADT object so that protein expression can be visualized on the same embedding.

In [ ]:
#| eval: false
adt_norm.obsm["X_umap"] = rna.obsm["X_umap"].copy()
adt_norm.obs = rna.obs.copy()

Inspect available ADT markers. Depending on the dataset, marker names may differ slightly from standard gene symbols.

In [ ]:
#| eval: false
adt_norm.var_names.to_list()

In [ ]:
#| eval: false
adt_markers = [
    "CD3", "CD4", "CD8", "CD14", "CD16", "CD19", "CD20",
    "CD56", "CD11c", "CD123", "HLA-DR", "CD45RA", "CD45RO",
]

adt_markers_present = [
    marker for marker in adt_markers
    if marker in adt_norm.var_names
]

adt_markers_present

In [ ]:
#| eval: false
# if exact names differ:

[x for x in adt_norm.var_names if "CD3" in x]

In [ ]:
#| eval: false
# Replace with ADT markers actually present in the object
adt_markers_to_plot = adt_markers_present[:12]

sc.pl.umap(
    adt_norm,
    color=adt_markers_to_plot,
    frameon=False,
    ncols=4,
    vmax="p99",
)

### Task 4: ADT-supported annotation

Use ADT markers to evaluate your RNA-based cell-type labels.

Answer the following:

1. Which RNA-based annotations are clearly supported by ADT markers?
2. Are there clusters where ADT markers suggest a different or more specific identity?
3. Give one example where protein information is especially helpful.

**Your interpretation:**  
Write 5–8 sentences here.

## Compare RNA and ADT evidence

Choose two cell populations and compare RNA and ADT evidence for their identity.

Examples:

- CD4 T cells versus CD8 T cells
- NK cells versus cytotoxic T cells
- CD14 monocytes versus FCGR3A/CD16 monocytes
- B cells versus plasma cells

In [ ]:
#| eval: false
# Example: visualize manual labels together with selected RNA/ADT markers
sc.pl.umap(rna, color=["manual_cell_type"], frameon=False)

In [ ]:
#| eval: false
# RNA marker example
sc.pl.umap(
    rna,
    color=[g for g in ["CD3D", "CD4", "CD8A", "MS4A1", "LYZ", "NKG7"] if g in rna.var_names],
    frameon=False,
    ncols=3,
    vmax="p99",
)

In [ ]:
#| eval: false
# ADT marker example
sc.pl.umap(
    adt_norm,
    color=[g for g in ["CD3", "CD4", "CD8", "CD19", "CD14", "CD56"] if g in adt_norm.var_names],
    frameon=False,
    ncols=3,
    vmax="p99",
)

### Task 5: RNA versus protein interpretation

Select two immune populations and explain how RNA and ADT information support their annotation.

For each population, mention:

- one RNA marker;
- one ADT marker;
- whether the two modalities agree;
- whether one modality is more informative than the other.

**Your interpretation:**  
Write your answer here.

## COVID-19 interpretation

Use the disease or severity metadata to explore whether cell-type composition differs between groups.

This does not need to be a formal differential abundance analysis. A simple normalized composition plot is sufficient.

In [ ]:
#| eval: false
# Replace with your actual metadata column names
group_col = "severity"  # e.g. "disease_status", "severity", "Status"
celltype_col = "manual_cell_type"

if group_col in rna.obs.columns:
    composition = (
        rna.obs
        .groupby([group_col, celltype_col])
        .size()
        .reset_index(name="n")
    )

    composition["fraction"] = (
        composition["n"] /
        composition.groupby(group_col)["n"].transform("sum")
    )

    composition.head()
else:
    print(f"{group_col!r} not found in rna.obs. Choose another metadata column.")

In [ ]:
#| eval: false
if group_col in rna.obs.columns:
    pivot = composition.pivot(
        index=group_col,
        columns=celltype_col,
        values="fraction",
    ).fillna(0)

    pivot.plot(kind="bar", stacked=True, figsize=(8, 4))
    plt.ylabel("Fraction of cells")
    plt.xlabel(group_col)
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()

### Task 6: Biological interpretation

Based on your plots, discuss whether the relative abundance of broad immune populations appears to differ between COVID-19 groups.

You do not need to make strong causal claims. Focus on what is visible in the sketched dataset and mention limitations.

**Your interpretation:**  
Write 5–8 sentences here.

## Final answer

Summarize your analysis in one short final paragraph.

Your summary should include:

1. the main cell populations you identified;
2. one example where ADT helped the interpretation;
3. one observation related to COVID-19 disease status or severity;
4. one limitation of the analysis.

**Final summary:**  
Write your final answer here.